# FLOWER noise mismatch benchmark

This notebook runs the exact project protocol on a Colab GPU: 20 fixed CelebA images, 20 fixed AFHQ-Cat images, five inverse problems, and five assumed noise ratios. The 1.0 ratio is the correctly specified control. The measurement, degradation operator, and solver random trajectory stay fixed across ratios. Results are stored in Google Drive and the runner resumes completed cases after a disconnect.

## Before running
Choose **Runtime > Change runtime type > T4 GPU**. Then run all cells. Google will ask for permission when Drive is mounted.

In [ ]:
import subprocess, sys
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. Select Runtime > Change runtime type > T4 GPU.')
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, subprocess
repo_url = 'https://github.com/ItayShorian07/Diffusion--flower-project.git'
repo_dir = Path('/content/Diffusion--flower-project')
if repo_dir.exists():
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Repository:', repo_dir)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
subprocess.run(['bash', 'scripts/download_benchmark_models.sh'], check=True)
subprocess.run(['bash', 'scripts/prepare_benchmark_data.sh'], check=True)

In [ ]:
results_dir = Path('/content/drive/MyDrive/RobustFLOWER/main')
figures_dir = Path('/content/drive/MyDrive/RobustFLOWER/figures')
results_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)
print('Persistent results directory:', results_dir)

## Full GPU run
This cell executes all 1,000 reconstructions with the paper's task-specific FLOWER step counts. Re-run the cell after a Colab disconnect. Existing rows are detected and skipped.

In [ ]:
command = [
    sys.executable, 'experiments/run_noise_mismatch.py',
    '--device', 'cuda',
    '--image-batch-size', '1',
    '--output', str(results_dir),
]
subprocess.run(command, check=True)

In [ ]:
subprocess.run([sys.executable, 'experiments/summarize_results.py', '--input', str(results_dir / 'metrics.csv'), '--output', str(results_dir)], check=True)
subprocess.run([sys.executable, 'experiments/capture_analysis.py', '--device', 'cuda', '--input', str(results_dir)], check=True)
subprocess.run([sys.executable, 'experiments/plot_results.py', '--input', str(results_dir), '--output', str(figures_dir)], check=True)

In [ ]:
import shutil
archive_base = '/content/drive/MyDrive/RobustFLOWER/robust_flower_results'
archive = shutil.make_archive(archive_base, 'zip', '/content/drive/MyDrive/RobustFLOWER')
print('Completed rows:', sum(1 for _ in open(results_dir / 'metrics.csv')) - 1)
print('Figures:', sorted(path.name for path in figures_dir.glob('*.png')))
print('Archive:', archive)